**The section below is for merging the basic info: CVE, outbreak, phr, enrollment for K12 students at county level**

The base data can be referenced from data folder. We will merge the data based on county name.

Notice: outbreak data other than 2025 is documented in binary way. If needs to do time series analysis, we will change the data to the actual count

In [1]:
library(tidyverse)
library(dplyr)
outbreak <- read.csv("data/raw/base/count.csv")
cve <- read.csv("data/raw/base/cve.csv")
phr_conv <- read.csv("data/raw/brfss/phr_conversion.csv")
df <- cve %>%
  inner_join(outbreak, by = "County",
             suffix = c("_cve", "_out"))
head(df)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


,County,X2016_cve,X2017_cve,X2018_cve,X2019_cve,X2020_cve,X2021_cve,X2022_cve,X2023_cve,X2024_cve,⋯,X2016_out,X2017_out,X2018_out,X2019_out,X2020_out,X2021_out,X2022_out,X2023_out,X2024_out,X2025_out
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,Anderson,0.35%,0.50%,0.75%,0.94%,1.00%,1.11%,1.62%,1.64%,2.05%,⋯,0,0,0,0,0,0,0,0,0,0
2,Andrews,0.75%,0.90%,1.07%,1.50%,0.39%,1.36%,0.45%,1.57%,1.43%,⋯,0,0,0,0,0,0,0,0,0,3
3,Angelina,0.70%,0.63%,0.67%,0.82%,0.96%,0.86%,1.03%,1.53%,1.98%,⋯,0,0,0,0,0,0,0,0,0,0
4,Aransas,1.22%,1.39%,1.49%,1.71%,1.57%,1.51%,0.00%,1.91%,1.91%,⋯,0,0,0,0,0,0,0,0,0,0
5,Archer,0.61%,0.45%,0.81%,1.43%,1.29%,1.56%,1.68%,2.32%,2.80%,⋯,0,0,0,0,0,0,0,0,0,0
6,Armstrong,0.59%,1.90%,1.44%,2.35%,3.53%,3.53%,3.77%,4.46%,4.95%,⋯,0,0,0,0,0,0,0,0,0,0


In [8]:
df2025 <- df %>%
  transmute(
    County,
    cve = `X2025_cve`,
    outbreak = `X2025_out`
  )
head(df2025)

,County,cve,outbreak
,<chr>,<chr>,<int>
1,Anderson,2.54%,0
2,Andrews,1.91%,3
3,Angelina,2.50%,0
4,Aransas,2.06%,0
5,Archer,2.70%,0
6,Armstrong,5.24%,0


In [9]:
df2025$cve <- gsub("%", "", df2025$cve)
df2025$cve <- as.numeric(df2025$cve)
head(df2025)
summary(df2025)

,County,cve,outbreak
,<chr>,<dbl>,<int>
1,Anderson,2.54,0
2,Andrews,1.91,3
3,Angelina,2.50,0
4,Aransas,2.06,0
5,Archer,2.70,0
6,Armstrong,5.24,0


    County               cve            outbreak      
 Length:254         Min.   : 0.000   Min.   :  0.000  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000  
 Mode  :character   Median : 2.490   Median :  0.000  
                    Mean   : 2.817   Mean   :  2.925  
                    3rd Qu.: 3.765   3rd Qu.:  0.000  
                    Max.   :14.540   Max.   :414.000  

In [10]:
enrollment <- read.csv("data/raw/base/enrollment.csv")
population <- read.csv("data/raw/base/population.csv")
enrollment <- enrollment %>%
  mutate(
    County = County %>%
      str_to_lower() %>%
      str_remove(" county") %>%
      str_trim() %>%
      str_to_title()
  )
enrollment <- enrollment[enrollment$County != "Grand Total", ]
View(enrollment)

,County,Enrollment.Sum
,<chr>,<int>
1,Anderson,7808
2,Andrews,4209
3,Angelina,15649
4,Aransas,2913
5,Archer,2110
6,Armstrong,297
7,Atascosa,9046
8,Austin,6290
9,Bailey,1330


In [11]:
df2025 <- df2025 %>%
  left_join(enrollment, by = "County") %>%
  transmute(
    County,
    cve,
    outbreak,
    enrollment = Enrollment.Sum
  )
df2025 <- df2025 %>%
  left_join(population, by = "County") %>%
  transmute(
    County,
    cve,
    outbreak,
    enrollment,
    population = population$X2025_pop
  )

In [12]:
head(df2025)

,County,cve,outbreak,enrollment,population
,<chr>,<dbl>,<int>,<int>,<int>
1,Anderson,2.54,0,7808,60303
2,Andrews,1.91,3,4209,19823
3,Angelina,2.50,0,15649,87492
4,Aransas,2.06,0,2913,25943
5,Archer,2.70,0,2110,8976
6,Armstrong,5.24,0,297,1806


In [13]:
df2025 <- df2025 %>%
  inner_join(phr_conv, by = "County") %>%
    transmute(
      County,
      cve,
      outbreak,
      enrollment,
      population,
      phr = PHR
    )
head(df2025)
summary(df2025)

,County,cve,outbreak,enrollment,population,phr
,<chr>,<dbl>,<int>,<int>,<int>,<int>
1,Anderson,2.54,0,7808,60303,4
2,Andrews,1.91,3,4209,19823,9
3,Angelina,2.50,0,15649,87492,5
4,Aransas,2.06,0,2913,25943,11
5,Archer,2.70,0,2110,8976,2
6,Armstrong,5.24,0,297,1806,1


    County               cve            outbreak         enrollment    
 Length:254         Min.   : 0.000   Min.   :  0.000   Min.   :    93  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000   1st Qu.:  1180  
 Mode  :character   Median : 2.490   Median :  0.000   Median :  3240  
                    Mean   : 2.817   Mean   :  2.925   Mean   : 22023  
                    3rd Qu.: 3.765   3rd Qu.:  0.000   3rd Qu.: 10549  
                    Max.   :14.540   Max.   :414.000   Max.   :879002  
                                                       NA's   :5       
   population           phr        
 Min.   :     55   Min.   : 1.000  
 1st Qu.:   6389   1st Qu.: 2.000  
 Median :  19718   Median : 5.000  
 Mean   : 124300   Mean   : 5.323  
 3rd Qu.:  56581   3rd Qu.: 8.000  
 Max.   :5003892   Max.   :11.000  
                                   

In [14]:
write.csv(df2025, file = "data/county_data.csv", row.names = FALSE)

**This section is for cleaning the general census data in texas for each county up to need**

In [15]:
# ----- Pull all county-level variables for Texas -----
library(viridis)
library(tidycensus)
census_api_key("d57d54c0ecada0a165fe2386aa8f5078acb68d5b")
texas_counties <- get_acs(
  geography = "county",
  state     = "TX",
  year      = 2024,       
  survey    = "acs5",     
  variables = c(
    # Demographics
    pct_hispanic     = "DP05_0090PE",  # % Hispanic
    pct_black        = "DP05_0045PE",  # % Black non-Hispanic
    pct_white        = "DP05_0037PE",  # % White non-Hispanic
    
    # Socioeconomic
    pct_poverty      = "DP03_0119PE",  # % below poverty line
    pct_uninsured    = "DP03_0099PE",  # % no health insurance
    pct_college      = "DP02_0068PE",  # % college degree+
    median_income    = "DP03_0062PE",   # median household income

    
    # Rural/accessx
    pct_foreign_born = "DP02_0093PE"   # % foreign born
  ),
  output = "wide"   # puts each variable in its own column
)
# Clean it up
texas_counties <- texas_counties %>%
  # Extract county FIPS code
  mutate(county_fips = substr(GEOID, 3, 5)) %>%
  # Keep only the estimate columns (drop margin of error)
  select(GEOID, NAME, county_fips,
         pct_hispanic, pct_black, pct_white,
         pct_poverty, pct_uninsured, 
         pct_college, median_income,
         pct_foreign_born)
head(texas_counties)
summary(texas_counties)

Loading required package: viridisLite

To install your API key for use in future sessions, run this function with `install = TRUE`.

Getting data from the 2020-2024 5-year ACS

Using the ACS Data Profile



GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
48001,"Anderson County, Texas",001,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1
48003,"Andrews County, Texas",003,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2
48005,"Angelina County, Texas",005,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2
48007,"Aransas County, Texas",007,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5
48009,"Archer County, Texas",009,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8
48011,"Armstrong County, Texas",011,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5


    GEOID               NAME           county_fips         pct_hispanic  
 Length:254         Length:254         Length:254         Min.   : 0.00  
 Class :character   Class :character   Class :character   1st Qu.:19.68  
 Mode  :character   Mode  :character   Mode  :character   Median :27.80  
                                                          Mean   :35.68  
                                                          3rd Qu.:50.02  
                                                          Max.   :97.20  
                                                                         
   pct_black        pct_white      pct_poverty    pct_uninsured  
 Min.   : 0.000   Min.   :14.40   Min.   : 0.00   Min.   : 0.00  
 1st Qu.: 1.200   1st Qu.:55.17   1st Qu.: 7.50   1st Qu.:13.90  
 Median : 3.950   Median :65.80   Median :10.80   Median :16.70  
 Mean   : 6.007   Mean   :63.84   Mean   :11.25   Mean   :16.97  
 3rd Qu.: 8.200   3rd Qu.:75.97   3rd Qu.:13.88   3rd Qu.:19.30  
 Max.   :33.

In [16]:
texas_counties <- texas_counties %>%
  mutate(
    County = NAME %>%
      str_to_lower() %>%
      str_remove(" county, texas") %>%
      str_trim() %>%
      str_to_title()
  )
head(texas_counties)

GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born,County
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
48001,"Anderson County, Texas",001,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1,Anderson
48003,"Andrews County, Texas",003,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2,Andrews
48005,"Angelina County, Texas",005,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2,Angelina
48007,"Aransas County, Texas",007,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5,Aransas
48009,"Archer County, Texas",009,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8,Archer
48011,"Armstrong County, Texas",011,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5,Armstrong


In [17]:
texas_counties <- texas_counties[, !names(texas_counties) %in% c("GEOID", "NAME", "county_fips")]

texas_counties <- texas_counties %>% relocate(County)
head(texas_counties)
write.csv(texas_counties, file = "data/texas_census.csv", row.names = FALSE)

County,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Anderson,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1
Andrews,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2
Angelina,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2
Aransas,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5
Archer,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8
Armstrong,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5


**This section is for cleaning BRFSS data.**
What we did was from the Total row, and col D10, F10,... for the response and extract as sheetname-response.

In [ ]:
import re
import pandas as pd
from openpyxl import load_workbook
from pathlib import Path
from collections import defaultdict

# CONFIGURE
SRC_DIR      = "data/raw/brfss"
SCRAPED_2014 = "data/created/brfss_scraped_2014.csv"   
OUT          = "data/brfss_all_categories.csv"

SCRAPED_PHRS  = {1, 8, 11}   # 2014 values for these PHRs come from the scrape, not Excel
SCRAPED_SCALE = 1.0          # multiply scraped % to match Excel's scale.

PCT_COLS = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23]


def clean(x):
    x = re.sub("[^A-Za-z0-9]+", "_", str(x))
    x = re.sub("^_+|_+$", "", x)
    return x.lower()


# ── Load scraped 2014 (PHR 1/8/11) -> {variable_key: {phr: pct}} ──────────────
# brfss_scraped_2014.csv is LONG format: one row per (variable, phr, percent).
scraped_df = pd.read_csv(SCRAPED_2014)
scraped_2014 = {}
for _, r in scraped_df.iterrows():
    phr = int(r["phr"])
    if phr in SCRAPED_PHRS and pd.notna(r["percent"]):
        scraped_2014.setdefault(str(r["variable"]), {})[phr] = float(r["percent"]) * SCRAPED_SCALE


records_2014 = {}   # (sheet, response) -> {phr: pct}  Excel only; skips PHR 1/8/11
records_2024 = {}

for phr in range(1, 12):
    for year, records in [(2014, records_2014), (2024, records_2024)]:
        # 2014 PHR 1/8/11 come from the scrape -> don't read those Excel files
        if year == 2014 and phr in SCRAPED_PHRS:
            continue
        path = f"{SRC_DIR}/{year}_PHR{phr}_BRFSS_Summary_Tables.xlsx"
        wb   = load_workbook(path, read_only=True)
        for sheet_name in wb.sheetnames:
            if sheet_name == "Index":
                continue
            rows = list(wb[sheet_name].iter_rows(values_only=True))
            if len(rows) < 10:
                continue
            label_row = rows[7]   # row 8  — response labels
            total_row = rows[9]   # row 10 — Total row
            for col in PCT_COLS:
                label = label_row[col] if col < len(label_row) else None
                pct   = total_row[col] if col < len(total_row) else None
                if not label or str(label).strip() in ("", "None"):
                    continue
                key = (sheet_name, str(label).strip())
                records.setdefault(key, {})[phr] = pct
    print(f"PHR {phr:2d} done")


# ── Match 2014 Excel names to the 2024 names (PHR 2-7,9,10) ────────────────────
# The 2014 workbooks use a DIFFERENT (and internally inconsistent) naming than
# 2024 — e.g. 2024 "A One C" is 2014 "A1C Test the Past Year"; 2024 "Heavy
# Drinking - Males" (answers Yes/No) is 2014 "Heavy Alc. Consumption Males"
# (answers At Risk/Not At Risk). A plain exact-key lookup therefore misses ~80%
# of variables. We normalize names (expand abbreviations, drop filler tokens),
# add a curated override for the semantic/abbreviated cases, then match the
# response inside the matched feature (exact -> synonym -> token-containment).
# NOTE: variables with no 2014 counterpart at all (e-cigarettes, BP checks, Rx
# pain meds, "Advised to ...", Alzheimer's, cervical-screening composites, the
# "IA" variants, etc.) are 2024-only and cannot have a 2014->2024 difference.
_ABBR = {"yr": "year", "yrs": "years", "sigmd": "sigmoidoscopy", "sigm": "sigmoidoscopy",
         "hlth": "health", "consump": "consumption", "alc": "alcohol", "dr": "doctor",
         "ed": "education", "educ": "education", "fem": "females", "activ": "activities",
         "diab": "diabetes"}
_FILL = {"the", "a", "an", "of", "in", "for", "and", "or", "to", "is", "w", "bc", "at", "age", "past"}

def _tok(s):
    s = re.sub("[^a-z0-9]+", " ", str(s).lower())
    return [_ABBR.get(t, t) for t in s.split() if t not in _FILL]

def _kf(s):
    return " ".join(_tok(s))

def _overlap(a, b):
    A, B = set(_tok(a)), set(_tok(b))
    return len(A & B) / max(len(A), len(B)) if A and B else 0

# 2024 feature -> 2014 Excel feature, only for cases normalization can't bridge.
FEATURE_OVERRIDE = {
    "A One C": "A1C Test the Past Year",
    "Binge Drinking": "Binge Drinking Past Month",
    "Binge-Heavy Drinking - Femal": "Binge Heavy Alcohol Fem 18-44",
    "CVD": "Cardiovascular Disease",
    "ClnscpySgmscpy": "Most Recent Exam Sigmd or Col",
    "Current Asthma 2": "Current Asthma Status",
    "Depressive Disorder": "Depressive Disorders",
    "Diabetes Age": "Age When Diagnosed w Diabetes",
    "Diabetes Education": "Ever Attended Diabetes Educ",
    "Diabetes Eye Exam": "Time Since Last Eye Exam",
    "Diabetes Test": "Tested for Diabetes",
    "Difficulty Doing Errands Alo": "Difficulty Doing Errands Alone",
    "Difficulty Dressing": "Difficulty Dressing or Bathing",
    "DurationClnscpySgmy": "Time Since Last Sigmd or Col",
    "DurationHomeBloodStoolTest": "Time Since Blood Stool Test",
    "Ever Had Sgmscpy-clnscpy": "Ever Had Sigmd or Colonoscopy",
    "FOBT Past Yr": "FOBT Past Yr Age 50-75",
    "Flu Shot, 18-64": "Flu Shot Past Year Age 18-64",
    "Flu Shot, 65+": "Flu Shot Past Year Age 65+",
    "Flu Vaccine Location": "Location of Flu Shot",
    "Have At Least One Personal D": "Have At Least One Personal Dr",
    "Health Care Coverage Age 18-": "Health Care Coverage Age 18-64",
    "Health Care Provider": "Have Personal Doctor",
    "Heavy Drinking": "Heavy Alcohol Consumption",
    "Heavy Drinking - Females": "Heavy Alcohol Consumption Females",
    "Heavy Drinking - Males": "Heavy Alcohol Consumption Males",
    "Hlth Affected Activ 14+ Days": "Poor Hlth Affected Activ 14+",
    "Hlth Affected Activ 5+ Days": "Poor Hlth Affected Activ 5+",
    "Hysterectomy": "Had Hysterectomy",
    "Insulin": "Taking Insulin",
    "Last Dentist Visit": "Time Since Last Dentist Visit",
    "Last Smoked": "Time Since Quit Smoking",
    "Mammogram": "Ever Had Mammogram",
    "Pap Test Past 3 Yrs 21-65": "Pap Smear Past 3 Yrs Age 21-65",
    "Pneumonia Shot, 18+": "Ever Had Pneumonia Shot",
    "Pneumonia Shot, 65+": "Ever Had Pneumonia Shot Age 65+",
    "Poor Physical Health 14+ Day": "Poor Physical Health 14+ Days",
    "Pre-Diabetes": "Prediabetes",
    "RemoveTeeth": "Permanent Teeth Removed",
    "Sigm and Blood Stool Age 50-": "Sigmd 5 Yrs FOBT 3 Yrs 50-75",
    "Up-To-Date CRC Scrn Age 50-7": "Up-To-Date CRC Scrn Age 50-75",
}
# 2024 response -> 2014 response, for non-lexical wordings (only fires when the
# target is actually one of the matched feature's responses).
RESPONSE_SYNONYM = {
    "recommended range": "Normal weight",                 # BMI categories
    "yes": "At Risk", "no": "Not At Risk",                # Heavy Drinking
    "40 years or younger": "Less than 40 years old",      # Diabetes Age
    "41 to 64": "40 to 64 years old",                     # Diabetes Age
}

# index the 2014 Excel records by normalized feature (merges the year's own name variants)
records_2014_by_feature = defaultdict(lambda: defaultdict(dict))
for (sheet_name, response), phr_vals in records_2014.items():
    records_2014_by_feature[_kf(sheet_name)][response].update(phr_vals)

# map each 2024 feature to a 2014 normalized-feature key (override, else exact-normalized)
excel_feature_map = {}
for sheet_name in {k[0] for k in records_2024}:
    fk = _kf(FEATURE_OVERRIDE.get(sheet_name, sheet_name))
    if fk in records_2014_by_feature:
        excel_feature_map[sheet_name] = fk

def _match_2014_response(response, resps):
    target = _kf(response)
    for rr in resps:                                   # exact (normalized)
        if _kf(rr) == target:
            return rr
    syn = RESPONSE_SYNONYM.get(target)                 # non-lexical synonym
    if syn:
        sk = _kf(syn)
        for rr in resps:
            if _kf(rr) == sk:
                return rr
    rtoks = set(_tok(response)); best = None; fewest = 1e9   # 2024 tokens ⊆ 2014 tokens
    for rr in resps:
        cand = set(_tok(rr))
        if rtoks and rtoks <= cand and len(cand - rtoks) < fewest:
            fewest = len(cand - rtoks); best = rr
    if best:
        return best
    scored = max(((_overlap(response, rr), rr) for rr in resps), default=(0, None))
    return scored[1] if scored[0] >= 0.8 else None


def value_2014(sheet_name, response, phr):
    """2014 value: scrape for PHR 1/8/11, name-matched Excel for the other PHRs."""
    if phr in SCRAPED_PHRS:
        return scraped_2014.get(clean(f"{sheet_name} - {response}"), {}).get(phr)
    fk = excel_feature_map.get(sheet_name)
    if not fk:
        return None
    rr = _match_2014_response(response, records_2014_by_feature[fk])
    return records_2014_by_feature[fk].get(rr, {}).get(phr) if rr else None


rows_out = []
for (sheet_name, response), phr_vals in records_2024.items():
    row = {"Sheet": f"{sheet_name} - {response}"}

    # 2024 PHR columns
    for phr in range(1, 12):
        row[str(phr)] = phr_vals.get(phr, None)

    # percent-difference columns (2024 vs 2014)
    for phr in range(1, 12):
        v14 = value_2014(sheet_name, response, phr)
        v24 = phr_vals.get(phr)
        try:
            pct_diff = round((float(v24) - float(v14)) / abs(float(v14)) * 100, 4)
        except (TypeError, ValueError, ZeroDivisionError):
            pct_diff = None
        row[f"PHR{phr}_pct_diff"] = pct_diff

    rows_out.append(row)


df = pd.DataFrame(rows_out).sort_values("Sheet").reset_index(drop=True)
df.to_csv(OUT, index=False)
n_diff = sum(df[f"PHR{p}_pct_diff"].notna().sum() for p in range(1, 12))
print(f"\nSaved: {OUT}  ({len(df):,} rows x {len(df.columns)} cols; {n_diff:,} non-NA pct_diff cells)")

**This section is for merging all the data as a file**
The file is with base info, full BRFSS data(with total row), and census data. We will join SVI data if needed.

In [2]:
county_data <- read_csv("data/county_data.csv") %>%
  rename(PHR = phr) %>%
  mutate(cve = as.numeric(str_remove(cve, "%")))

write_csv(county_data, "data/county_data.csv")

Rows: 254 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): County
dbl (5): cve, outbreak, enrollment, population, PHR

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


ERROR: [1m[33mError[39m in `rename()`:[22m
[33m![39m Can't rename columns that don't exist.
[31m✖[39m Column `phr` doesn't exist.


In [3]:
library(dplyr)
library(tidyr)
library(readr)
library(stringr)

county_data      <- read_csv("data/county_data.csv")
texas_census     <- read_csv("data/texas_census.csv")
brfss_wide       <- read_csv("data/brfss_all_categories.csv")

normalize_county <- function(x) str_to_title(x)

county_data  <- county_data  |> mutate(County = normalize_county(County))
texas_census <- texas_census |> mutate(County = normalize_county(County))

county_merged <- county_data |>
  left_join(texas_census, by = "County")

cat("county_data rows:     ", nrow(county_data), "\n")
cat("After county join:    ", nrow(county_merged), "\n")
cat("Unmatched counties:   ",
    sum(is.na(county_merged$pct_hispanic)), "\n")

brfss_renamed <- brfss_wide |> rename(variable = Sheet)

# 2024 PHR percentages  (columns "1".."11")
pct_wide <- brfss_renamed |>
  select(variable, all_of(as.character(1:11))) |>
  pivot_longer(-variable, names_to = "PHR", values_to = "value") |>
  mutate(PHR   = as.integer(PHR),
         value = suppressWarnings(as.numeric(value))) |>
  pivot_wider(names_from = variable, values_from = value)

# 2014 -> 2024 percent difference  (columns "PHR1_pct_diff".."PHR11_pct_diff")
diff_wide <- brfss_renamed |>
  select(variable, ends_with("_pct_diff")) |>
  pivot_longer(-variable, names_to = "PHR", values_to = "pct_diff") |>
  mutate(PHR      = as.integer(str_extract(PHR, "[0-9]+")),
         pct_diff = suppressWarnings(as.numeric(pct_diff))) |>
  pivot_wider(names_from = variable, values_from = pct_diff,
              names_glue = "{variable}_pct_diff")

brfss_county <- pct_wide |> left_join(diff_wide, by = "PHR")

final <- county_merged |> left_join(brfss_county, by = "PHR")

cat("Final dimensions:     ", nrow(final), "rows x", ncol(final), "cols\n")

write_csv(final, "/Users/sean/Summer 2026 Research/Measles-Outbreak-and-Public-Policy-Reluctance/2026SU/data/merged_data.csv")

Rows: 254 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): County
dbl (5): cve, outbreak, enrollment, population, PHR

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 254 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): County
dbl (7): pct_hispanic, pct_black, pct_white, pct_poverty, pct_uninsured, pct...
lgl (1): median_income

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 391 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): Sheet, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11
dbl (11): PHR1_pct_diff, PHR2_pct_diff, PHR3_pct_diff, PHR4_pct_diff, PHR5_p...

ℹ Use `spec()` to r

county_data rows:      254 
After county join:     254 
Unmatched counties:    0 
Final dimensions:      254 rows x 796 cols


with SVI data

In [4]:
library(dplyr)
library(readr)

merged  <- read_csv("data/merged_data.csv")
svi     <- read_csv("data/svi_interactive_map.csv")

svi <- svi |>
  mutate(County = gsub(" County", "", COUNTY)) |>
  select(-ST, -STATE, -ST_ABBR, -STCNTY, -COUNTY, -FIPS, -LOCATION, -GeoLevel, -Comparison)

final <- merged |>
  left_join(svi, by = "County")

write_csv(final, "data/merged_with_svi.csv")

Rows: 254 Columns: 796
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (1): County
dbl (596): cve, outbreak, enrollment, population, PHR, pct_hispanic, pct_bla...
lgl (199): median_income, Advised to Cut Down Salt - Do not use salt, Diabet...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 254 Columns: 160
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (6): STATE, ST_ABBR, COUNTY, LOCATION, GeoLevel, Comparison
dbl (154): ST, STCNTY, FIPS, AREA_SQMI, E_TOTPOP, M_TOTPOP, E_HU, M_HU, E_HH...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [17]:

# test <- final %>% select(where(~ mean(is.na(.)) <= 0))  %>% select(ends_with("diff"))

# names(test)

[1] "Any Cancer - No_pct_diff"                                         
 [2] "Any Cancer - Yes_pct_diff"                                        
 [3] "Any Drinking Past Month - No_pct_diff"                            
 [4] "Any Drinking Past Month - Yes_pct_diff"                           
 [5] "Arthritis - No_pct_diff"                                          
 [6] "Arthritis - Yes_pct_diff"                                         
 [7] "Asthma Status - Never_pct_diff"                                   
 [8] "Binge Drinking - No_pct_diff"                                     
 [9] "Binge Drinking - Yes_pct_diff"                                    
[10] "Blind - No_pct_diff"                                              
[11] "COPD - No_pct_diff"                                               
[12] "CVD - No_pct_diff"                                                
[13] "ClnscpySgmscpy - Colonoscopy_pct_diff"                            
[14] "Col Past 10 Yrs 50-75 - No_pct_diff"                              
[15] "Col Past 10 Yrs 50-75 - Yes_pct_diff"                             
[16] "Coronary Heart Disease - No_pct_diff"                             
[17] "Current Asthma 2 - No_pct_diff"                                   
[18] "Current Smoker - No_pct_diff"                                     
[19] "Current Smoker - Yes_pct_diff"                                    
[20] "Dentist Visit Past Year - No_pct_diff"                            
[21] "Dentist Visit Past Year - Yes_pct_diff"                           
[22] "Depressive Disorder - No_pct_diff"                                
[23] "Depressive Disorder - Yes_pct_diff"                               
[24] "Diabetes - No_pct_diff"                                           
[25] "Diabetes - Yes_pct_diff"                                          
[26] "Diabetes Status - No_pct_diff"                                    
[27] "Diabetes Status - Yes_pct_diff"                                   
[28] "Ever Had Blood Stool Test - Yes_pct_diff"                         
[29] "Ever Had HIV Test - No_pct_diff"                                  
[30] "Ever Had HIV Test - Yes_pct_diff"                                 
[31] "Ever Smoked - No_pct_diff"                                        
[32] "Ever Smoked - Yes_pct_diff"                                       
[33] "FOBT Past Yr - No_pct_diff"                                       
[34] "Flu Shot in Past Year - No_pct_diff"                              
[35] "Flu Shot in Past Year - Yes_pct_diff"                             
[36] "Flu Shot, 18-64 - No_pct_diff"                                    
[37] "Flu Shot, 18-64 - Yes_pct_diff"                                   
[38] "Frequency of Smoking - Every day_pct_diff"                        
[39] "Frequency of Smoking - Not at all_pct_diff"                       
[40] "General Health - Excellent_pct_diff"                              
[41] "General Health - Fair_pct_diff"                                   
[42] "General Health - Good_pct_diff"                                   
[43] "General Health - Very Good_pct_diff"                              
[44] "Health Care Coverage - No_pct_diff"                               
[45] "Health Care Coverage - Yes_pct_diff"                              
[46] "Health Care Coverage Age 18- - No_pct_diff"                       
[47] "Health Care Coverage Age 18- - Yes_pct_diff"                      
[48] "Heart Attack - No_pct_diff"                                       
[49] "Heart Disease - No_pct_diff"                                      
[50] "Heavy Drinking - Females - No_pct_diff"                           
[51] "Heavy Drinking - Males - No_pct_diff"                             
[52] "Heavy Drinking - No_pct_diff"                                     
[53] "Hlth Affected Activ 14+ Days - 14 or more days_pct_diff"          
[54] "Hlth Affected Activ 14+ Days - None to less than 14 days_pct_diff"
[55] "Hlth Affected Activ 5+ Days - 5 or more days_pct_diff